# PDF Text Preprocessing for Next Word Prediction

PDF/Books → Clean Text → Tokenize → Save Dataset

In [6]:
import os
import re
import pymupdf
import fitz
import pickle
import json
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

os.makedirs("books", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

print("Folders ready")


Folders ready


## 1. Add Books

Put your PDF books inside the `books` folder.

In [7]:
books = [file for file in os.listdir("books") if file.endswith(".pdf")]

print("Books found:")
for book in books:
    print(book)

Books found:
10-Republic-author-Plato.pdf
Critique-of-Pure-Reason-Immanuel-Kant.pdf
How-to-win-friends-and-influence-people-Dale-Carnegie.pdf
on-the-origin-of-species-charles-darwin.pdf
pride-and-prejudice-jane-austen.pdf
think-and-grow-rich-napoleon-hill.pdf


In [8]:
def extract_text(pdf_file):
    doc = fitz.open(pdf_file)
    text = ""

    for page in doc:
        text += page.get_text() + "\n"

    doc.close()
    return text


all_text = ""

for book in books:
    text = extract_text("books/" + book)
    all_text += "\n" + text

print("Total characters:", len(all_text))
print(all_text[:1000])

Total characters: 4871555

The Republic
By Plato
http://www.idph.net
18 de maio de 2002

BOOK I
PERSONS OF THE DIALOGUE.
Socrates, who is the narrator.
Glaucon.
Adeimantus.
Polemarchus.
Cephalus.
Thrasymachus.
Cleitophon.
And others who are mute auditors.
The scene is laid in the house of Cephalus at the Piraeus; and the whole dia-
logue is narrated by Socrates the day after it actually took place to Timaeus,
Hermocrates, Critias, and a nameless person, who are introduced in the Tima-
eus.
I went down yesterday to the Piraeus with Glaucon the son of Ariston, that I
might offer up my prayers to the goddess (Bendis, the Thracian Artemis.); and
also because I wanted to see in what manner they would celebrate the festival,
which was a new thing. I was delighted with the procession of the inhabitants;
but that of the Thracians was equally, if not more, beautiful. When we had
ﬁnished our prayers and viewed the spectacle, we turned in the direction of the
city; and at that instant Polemarchus

## 2. Clean Text

In [9]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = text.strip()
    return text


cleaned_text = clean_text(all_text)

print("Characters after cleaning:", len(cleaned_text))
print(cleaned_text[:1000])

Characters after cleaning: 4815912
The Republic By Plato http://www.idph.net   de maio de   BOOK I PERSONS OF THE DIALOGUE. Socrates, who is the narrator. Glaucon. Adeimantus. Polemarchus. Cephalus. Thrasymachus. Cleitophon. And others who are mute auditors. The scene is laid in the house of Cephalus at the Piraeus; and the whole dia- logue is narrated by Socrates the day after it actually took place to Timaeus, Hermocrates, Critias, and a nameless person, who are introduced in the Tima- eus. I went down yesterday to the Piraeus with Glaucon the son of Ariston, that I might offer up my prayers to the goddess (Bendis, the Thracian Artemis.); and also because I wanted to see in what manner they would celebrate the festival, which was a new thing. I was delighted with the procession of the inhabitants; but that of the Thracians was equally, if not more, beautiful. When we had ﬁnished our prayers and viewed the spectacle, we turned in the direction of the city; and at that instant Polemarc

In [10]:
with open("processed_data/cleaned_text.txt", "w", encoding="utf-8") as file:
    file.write(cleaned_text)

print("Cleaned text saved")

Cleaned text saved


## 3. Split Text into Sentences

In [11]:
sentences = re.split(r"(?<=[.!?])\s+", cleaned_text)
sentences = [sentence.strip() for sentence in sentences if sentence.strip()]

print("Number of sentences:", len(sentences))
print(sentences[:5])

Number of sentences: 35223
['The Republic By Plato http://www.idph.net   de maio de   BOOK I PERSONS OF THE DIALOGUE.', 'Socrates, who is the narrator.', 'Glaucon.', 'Adeimantus.', 'Polemarchus.']


In [12]:
df = pd.DataFrame({"text": sentences})
df.to_csv("processed_data/sentences.csv", index=False)

print("Sentence dataset saved")

Sentence dataset saved


## 4. Tokenization

In [13]:
tokenizer = Tokenizer(
    num_words=30000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(sentences)

print("Vocabulary size:", len(tokenizer.word_index))

Vocabulary size: 26909


In [14]:
with open("processed_data/tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("Tokenizer saved")

Tokenizer saved


## 5. Convert Text into Numbers

In [15]:
sequences = tokenizer.texts_to_sequences(sentences)

print("First sentence:")
print(sentences[0])

print("\nTokenized:")
print(sequences[0])

First sentence:
The Republic By Plato http://www.idph.net   de maio de   BOOK I PERSONS OF THE DIALOGUE.

Tokenized:
[2, 7248, 23, 3559, 311, 303, 170, 308, 912, 14761, 912, 420, 14, 1175, 3, 2, 8128]


In [16]:
with open("processed_data/sequences.pkl", "wb") as file:
    pickle.dump(sequences, file)

print("Sequences saved")

Sequences saved


## 6. Create Next-Word Training Sequences

The previous 10 words are used to predict the next word.

In [17]:
sequence_length = 10

input_sequences = []

for sentence in sequences:
    for i in range(sequence_length, len(sentence)):
        input_sequences.append(sentence[i-sequence_length:i+1])

input_sequences = np.array(input_sequences)

print("Dataset shape:", input_sequences.shape)

Dataset shape: (551537, 11)


In [18]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

np.save("processed_data/X.npy", X)
np.save("processed_data/y.npy", y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (551537, 10)
y shape: (551537,)


## 7. Save Vocabulary

In [19]:
with open("processed_data/word_index.json", "w", encoding="utf-8") as file:
    json.dump(tokenizer.word_index, file)

print("Vocabulary saved")

Vocabulary saved


## Dataset Files

The `processed_data` folder contains the files needed for the LSTM training notebook.

In [20]:
print("Preprocessing completed successfully.")

print("\nFiles:")
for file in os.listdir("processed_data"):
    print(file)

Preprocessing completed successfully.

Files:
cleaned_text.txt
sentences.csv
sequences.pkl
tokenizer.pkl
word_index.json
X.npy
y.npy
